# Garbage Reports — Conformal Prediction-Interval Calibration

The selected point-forecasting model remains **`nb_recent3_holiday`**.

This notebook does **not** change the forecasting model. It calibrates the model's prediction intervals using a chronological split:

- **Training:** 2020–2023
- **Calibration:** 2024
- **Final untouched test:** 2025

The raw Negative Binomial interval is kept as a baseline.

For the conformal adjustment, each 2024 observation receives a nonconformity score:

\[
s_i = \max(L_i-y_i,\; y_i-U_i,\;0)
\]

where \(L_i,U_i\) are the raw Negative Binomial bounds.

The calibrated interval then becomes:

\[
[L-q,\;U+q]
\]

where \(q\) is chosen from the calibration scores using the finite-sample split-conformal quantile.

This preserves the shape of the Negative Binomial interval while widening it only as much as the calibration data suggests.

> Because this is time-series data rather than exchangeable IID data, the usual exact conformal coverage guarantee does not strictly apply. We therefore judge calibration empirically on the untouched 2025 period.


In [ ]:
import holidays
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error

from garbage_forecasting_holiday import (
    prepare_daily_data,
    fit_nb_recent3_holiday,
    nb_prediction_interval,
    classify_activity,
)


## 1. Configuration

Normally the only setting you may want to change later is `INTERVAL_LEVEL`.

Do **not** tune this value merely to force the 2025 coverage to equal a desired number; 2025 is our final test period in this notebook.


In [ ]:
PATH = r"data/requests.csv"

START_YEAR = 2020
END_YEAR = 2025

TRAIN_END_YEAR = 2023
CALIBRATION_YEAR = 2024
TEST_YEAR = 2025

INTERVAL_LEVEL = 0.95


## 2. Load and prepare the daily data


In [ ]:
raw_df = pd.read_csv(PATH)

daily_df = prepare_daily_data(
    raw_df,
    issue_type="garbage",
    start_year=START_YEAR,
    end_year=END_YEAR,
)

print(f"Daily observations: {len(daily_df):,}")
print(
    f"Date range: {daily_df['reported'].min().date()} "
    f"-> {daily_df['reported'].max().date()}"
)

print(
    f"Train through: {TRAIN_END_YEAR} | "
    f"Calibration: {CALIBRATION_YEAR} | "
    f"Final test: {TEST_YEAR}"
)


## 3. Fit on 2020–2023 and create raw 95% intervals for 2024

The 2024 observations are used **only to learn how much the raw Negative Binomial interval needs to be adjusted**.


In [ ]:
train = daily_df[
    daily_df["reported"].dt.year <= TRAIN_END_YEAR
].copy()

calibration = daily_df[
    daily_df["reported"].dt.year == CALIBRATION_YEAR
].dropna(subset=["recent3_avg"]).copy()

cal_model = fit_nb_recent3_holiday(train)

cal_mu, cal_lower, cal_upper = nb_prediction_interval(
    cal_model,
    calibration,
    level=INTERVAL_LEVEL,
)

calibration["prediction"] = cal_mu
calibration["raw_lower"] = cal_lower
calibration["raw_upper"] = cal_upper

calibration["raw_inside"] = (
    (calibration["reports"] >= calibration["raw_lower"])
    & (calibration["reports"] <= calibration["raw_upper"])
)

print(f"2024 raw coverage: {calibration['raw_inside'].mean():.2%}")
print(
    "2024 raw average width: "
    f"{(calibration['raw_upper'] - calibration['raw_lower']).mean():.2f}"
)


## 4. Learn the conformal widening amount from 2024

A score of 0 means the actual count was already inside the raw interval.

A positive score tells us how many reports outside the nearest bound the observation fell.

The finite-sample conformal quantile is calculated using:

\[
k = \lceil (n+1)(1-lpha)ceil
\]

and the \(k\)-th ordered calibration score.


In [ ]:
alpha = 1 - INTERVAL_LEVEL

calibration["nonconformity_score"] = np.maximum.reduce([
    calibration["raw_lower"].to_numpy() - calibration["reports"].to_numpy(),
    calibration["reports"].to_numpy() - calibration["raw_upper"].to_numpy(),
    np.zeros(len(calibration)),
])

scores = np.sort(
    calibration["nonconformity_score"].to_numpy(dtype=float)
)

n_cal = len(scores)

k = int(np.ceil((n_cal + 1) * (1 - alpha)))
k = min(max(k, 1), n_cal)

q_hat = scores[k - 1]

print(f"Calibration observations: {n_cal}")
print(f"Target interval level:    {INTERVAL_LEVEL:.1%}")
print(f"Conformal q_hat:          {q_hat:.1f} reports")
print()
print(
    "Interpretation: every raw NB interval will be widened by "
    f"{q_hat:.1f} report(s) on each side."
)

calibration["conformal_lower"] = np.maximum(
    0,
    calibration["raw_lower"] - q_hat
)

calibration["conformal_upper"] = (
    calibration["raw_upper"] + q_hat
)

calibration["conformal_inside"] = (
    (calibration["reports"] >= calibration["conformal_lower"])
    & (calibration["reports"] <= calibration["conformal_upper"])
)

print(
    f"2024 conformal coverage: "
    f"{calibration['conformal_inside'].mean():.2%}"
)


## 5. Final untouched evaluation on 2025

Now we:

1. refit `nb_recent3_holiday` using **2020–2024**;
2. forecast every day in **2025**;
3. create the ordinary Negative Binomial 95% interval;
4. apply the **fixed `q_hat` learned only from 2024**;
5. compare raw vs calibrated coverage on 2025.

No 2025 observation is used to choose the conformal adjustment.


In [ ]:
final_train = daily_df[
    daily_df["reported"].dt.year < TEST_YEAR
].copy()

test = daily_df[
    daily_df["reported"].dt.year == TEST_YEAR
].dropna(subset=["recent3_avg"]).copy()

final_model = fit_nb_recent3_holiday(final_train)

test_mu, test_raw_lower, test_raw_upper = nb_prediction_interval(
    final_model,
    test,
    level=INTERVAL_LEVEL,
)

test["prediction"] = test_mu
test["raw_lower"] = test_raw_lower
test["raw_upper"] = test_raw_upper

test["conformal_lower"] = np.maximum(
    0,
    test["raw_lower"] - q_hat
)

test["conformal_upper"] = (
    test["raw_upper"] + q_hat
)

test["raw_inside"] = (
    (test["reports"] >= test["raw_lower"])
    & (test["reports"] <= test["raw_upper"])
)

test["conformal_inside"] = (
    (test["reports"] >= test["conformal_lower"])
    & (test["reports"] <= test["conformal_upper"])
)

test["raw_status"] = classify_activity(
    test["reports"],
    test["raw_lower"],
    test["raw_upper"],
)

test["conformal_status"] = classify_activity(
    test["reports"],
    test["conformal_lower"],
    test["conformal_upper"],
)

mae = mean_absolute_error(
    test["reports"],
    test["prediction"],
)

rmse = np.sqrt(
    mean_squared_error(
        test["reports"],
        test["prediction"],
    )
)

summary = pd.DataFrame({
    "Method": [
        "Raw Negative Binomial",
        "Conformal-calibrated"
    ],
    "Coverage (%)": [
        100 * test["raw_inside"].mean(),
        100 * test["conformal_inside"].mean(),
    ],
    "Average Width": [
        (test["raw_upper"] - test["raw_lower"]).mean(),
        (
            test["conformal_upper"]
            - test["conformal_lower"]
        ).mean(),
    ],
    "Unusually Low": [
        (test["raw_status"] == "Unusually low").sum(),
        (test["conformal_status"] == "Unusually low").sum(),
    ],
    "Unusually High": [
        (test["raw_status"] == "Unusually high").sum(),
        (test["conformal_status"] == "Unusually high").sum(),
    ],
})

print("2025 POINT-FORECAST PERFORMANCE")
print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print()

print("2025 INTERVAL PERFORMANCE")
display(summary.round(2))


## 6. Visual comparison on 2025

The darker band is the raw Negative Binomial interval.

The lighter outer band is the conformal-calibrated interval.


In [ ]:
plt.figure(figsize=(15, 6))

plt.plot(
    test["reported"],
    test["reports"],
    linewidth=0.9,
    label="Actual",
)

plt.plot(
    test["reported"],
    test["prediction"],
    linewidth=1.5,
    label="Expected",
)

plt.fill_between(
    test["reported"],
    test["conformal_lower"],
    test["conformal_upper"],
    alpha=0.12,
    label="Conformal-calibrated interval",
)

plt.fill_between(
    test["reported"],
    test["raw_lower"],
    test["raw_upper"],
    alpha=0.25,
    label="Raw NB interval",
)

high = test["conformal_status"] == "Unusually high"
low = test["conformal_status"] == "Unusually low"

plt.scatter(
    test.loc[high, "reported"],
    test.loc[high, "reports"],
    marker="x",
    s=45,
    label="Unusually high after calibration",
)

plt.scatter(
    test.loc[low, "reported"],
    test.loc[low, "reports"],
    marker="x",
    s=45,
    label="Unusually low after calibration",
)

plt.xlabel("Date")
plt.ylabel("Garbage Reports")
plt.title(
    f"Raw vs Conformal-Calibrated "
    f"{INTERVAL_LEVEL:.0%} Prediction Intervals — {TEST_YEAR}"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Coverage by weekday and month on the untouched 2025 test set

This tells us whether calibration improves the weaker groups we previously noticed, especially weekends and some months.


In [ ]:
test["raw_width"] = (
    test["raw_upper"] - test["raw_lower"]
)

test["conformal_width"] = (
    test["conformal_upper"] - test["conformal_lower"]
)

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

weekday_compare = (
    test.groupby("weekday")
    .agg(
        Raw_Coverage=("raw_inside", "mean"),
        Conformal_Coverage=("conformal_inside", "mean"),
        Raw_Width=("raw_width", "mean"),
        Conformal_Width=("conformal_width", "mean"),
        N=("reports", "size"),
    )
    .reindex(weekday_order)
)

weekday_compare["Raw Coverage (%)"] = (
    100 * weekday_compare["Raw_Coverage"]
)
weekday_compare["Conformal Coverage (%)"] = (
    100 * weekday_compare["Conformal_Coverage"]
)

print("2025 COVERAGE BY WEEKDAY")
display(
    weekday_compare[
        [
            "Raw Coverage (%)",
            "Conformal Coverage (%)",
            "Raw_Width",
            "Conformal_Width",
            "N",
        ]
    ].round(2)
)

month_compare = (
    test.groupby("month")
    .agg(
        Raw_Coverage=("raw_inside", "mean"),
        Conformal_Coverage=("conformal_inside", "mean"),
        Raw_Width=("raw_width", "mean"),
        Conformal_Width=("conformal_width", "mean"),
        N=("reports", "size"),
    )
)

month_compare["Raw Coverage (%)"] = (
    100 * month_compare["Raw_Coverage"]
)
month_compare["Conformal Coverage (%)"] = (
    100 * month_compare["Conformal_Coverage"]
)

print("\n2025 COVERAGE BY MONTH")
display(
    month_compare[
        [
            "Raw Coverage (%)",
            "Conformal Coverage (%)",
            "Raw_Width",
            "Conformal_Width",
            "N",
        ]
    ].round(2)
)


## 8. Production interval for January 1, 2026

After the honest 2025 evaluation above, we may use **2025 as the most recent calibration year for deployment into 2026**.

For this production-only step:

1. fit a model through 2024 and generate out-of-sample raw intervals for 2025;
2. compute a fresh `q_hat_2025` from those 2025 scores;
3. fit the final production model on all data through 2025;
4. forecast January 1, 2026;
5. apply the 2025-derived conformal adjustment.

This does **not** change the 2025 evaluation shown above.


In [ ]:
# Step A: learn a production calibration adjustment from 2025.
prod_cal_train = daily_df[
    daily_df["reported"].dt.year < 2025
].copy()

prod_cal = daily_df[
    daily_df["reported"].dt.year == 2025
].dropna(subset=["recent3_avg"]).copy()

prod_cal_model = fit_nb_recent3_holiday(prod_cal_train)

_, prod_cal_lower, prod_cal_upper = nb_prediction_interval(
    prod_cal_model,
    prod_cal,
    level=INTERVAL_LEVEL,
)

prod_scores = np.maximum.reduce([
    prod_cal_lower - prod_cal["reports"].to_numpy(),
    prod_cal["reports"].to_numpy() - prod_cal_upper,
    np.zeros(len(prod_cal)),
])

prod_scores = np.sort(prod_scores.astype(float))

n_prod_cal = len(prod_scores)
k_prod = int(
    np.ceil((n_prod_cal + 1) * (1 - alpha))
)
k_prod = min(max(k_prod, 1), n_prod_cal)

q_hat_2025 = prod_scores[k_prod - 1]

# Step B: final model using all data through 2025.
production_model = fit_nb_recent3_holiday(daily_df)

next_date = daily_df["reported"].max() + pd.Timedelta(days=1)
trend_origin = daily_df["reported"].min()

future_holidays = holidays.country_holidays(
    "GR",
    years=[next_date.year],
)

next_row = pd.DataFrame({
    "reported": [next_date],
    "weekday": [next_date.day_name()],
    "month": [next_date.month],
    "trend_years": [
        (next_date - trend_origin).days / 365.25
    ],
    "recent3_avg": [
        daily_df["reports"].iloc[-3:].mean()
    ],
    "is_holiday": [
        next_date.date() in future_holidays
    ],
})

next_mu, next_raw_lower, next_raw_upper = (
    nb_prediction_interval(
        production_model,
        next_row,
        level=INTERVAL_LEVEL,
    )
)

next_conf_lower = max(
    0,
    int(next_raw_lower[0] - q_hat_2025)
)

next_conf_upper = int(
    next_raw_upper[0] + q_hat_2025
)

holiday_name = future_holidays.get(
    next_date.date(),
    ""
)

print(
    f"Holiday:                  "
    f"{holiday_name if holiday_name else 'No'}"
)
print(f"Forecast date:            {next_date.date()}")
print(f"Expected reports:         {next_mu[0]:.1f}")
print(
    f"Raw {INTERVAL_LEVEL:.0%} NB interval:      "
    f"[{next_raw_lower[0]}, {next_raw_upper[0]}]"
)
print(
    f"Production conformal q:   {q_hat_2025:.1f}"
)
print(
    f"Calibrated {INTERVAL_LEVEL:.0%} interval: "
    f"[{next_conf_lower}, {next_conf_upper}]"
)


## How to interpret the result

The most important comparison is the **2025 raw vs conformal coverage**.

A useful calibration should move observed coverage closer to the nominal target without making the intervals unreasonably wide.

If the conformal interval performs better on 2025, then the calibrated bounds are the more defensible thresholds for:

- **below lower bound** → unusually low;
- **inside interval** → normal;
- **above upper bound** → unusually high.

Do not judge calibration only by whether one specific day is classified correctly. The goal is reliable performance across the full unseen test period.
